In [6]:
import yfinance as yf
import pandas as pd
import numpy as np

# -----------------------
# Download adjusted prices, new public ETFs, and master calendar intersection
# -----------------------

start_date = "2014-01-01"
end_date   = "2026-01-01"

tickers = [
    "IVV", "IDEV", "AGG", "IAGG",
    "IYR", "IFGL", "IAU", "BTC-USD"
]

rename_map = {
    "IVV": "US_EQUITY",
    "IDEV": "INTL_EQUITY",
    "AGG": "US_BONDS",
    "IAGG": "INTL_BONDS",
    "IYR": "US_REIT",
    "IFGL": "INTL_REIT",
    "IAU": "GOLD",
    "BTC-USD": "BTC"
}

print("Downloading data...")

prices = (
    yf.download(
        tickers,
        start=start_date,
        end=end_date,
        auto_adjust=True,
        progress=False
    )["Close"]
    .rename(columns=rename_map)
    .sort_index()
)

prices = prices[list(rename_map.values())]

# -----------------------
# Master calendar (ETF intersection)
# -----------------------
etf_cols = [c for c in prices.columns if c != "BTC"]

first_common_start = prices[etf_cols].dropna(how="any").index.min()
prices = prices.loc[first_common_start:]

master_calendar = prices[etf_cols].dropna(how="any").index
master_calendar = master_calendar[master_calendar.dayofweek < 5]

# -----------------------
# Align prices
# -----------------------
raw_data = prices.reindex(master_calendar).dropna(how="any")

# -----------------------
# Compute log returns
# -----------------------
returns = np.log(raw_data / raw_data.shift(1)).dropna()

# -----------------------
# Save datasets
# -----------------------
raw_data.to_csv("raw_data.csv")
returns.to_csv("log_returns.csv")

print("Raw data:", raw_data.shape)
print("Returns:", returns.shape)
print("Date range:", returns.index.min().date(), "->", returns.index.max().date())

Raw data: (2205, 8)
Returns: (2204, 8)
Date range: 2017-03-28 -> 2025-12-31
